# BraTS 2024 Glioma MRI Preprocessing Pipeline

This notebook develops a reproducible preprocessing pipeline for the
BraTS 2024 Adult Glioma dataset in preparation for 3D deep-learning
segmentation.

The preceding exploratory notebook examined individual NIfTI volumes,
multimodal MRI alignment, reference tumour segmentations, MRI intensity
distributions and modality-wise normalization.

This notebook extends those concepts to the complete dataset using MONAI.

## Objectives

1. Discover all available BraTS training cases.
2. Construct a dataset manifest linking each case to its four MRI
   modalities and reference segmentation.
3. Validate dataset completeness before model training.
4. Define reproducible training and validation partitions.
5. Load multimodal NIfTI data using MONAI.
6. Apply spatial and intensity preprocessing.
7. Prepare segmentation targets for supervised learning.
8. Develop an appropriate 3D cropping and patch-sampling strategy.
9. Introduce training-specific data augmentation.
10. Construct MONAI datasets and PyTorch DataLoaders.
11. Inspect model-ready batches before beginning model training.

## 1. Environment and Dataset Configuration

The preprocessing pipeline begins by importing the required libraries and
defining the locations of the project repository and externally stored
BraTS dataset.

The BraTS imaging data are intentionally stored outside the Git repository
and are not version-controlled.

In [1]:
# -------------------------------------------------------------
# IMPORT LIBRARIES
# -------------------------------------------------------------
#
# Standard Python and scientific-computing libraries are used
# alongside MONAI, which provides medical-imaging-specific
# preprocessing and deep-learning utilities.

from pathlib import Path

import numpy as np
import torch
import monai

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)
print("MONAI version:", monai.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

NumPy version: 2.4.6
PyTorch version: 2.14.1+cu130
MONAI version: 1.6.1
CUDA available: True
GPU: NVIDIA GeForce RTX 4080 SUPER


In [2]:
# -------------------------------------------------------------
# DEFINE PROJECT AND DATASET PATHS
# -------------------------------------------------------------
#
# PROJECT_ROOT points to the Git repository.
#
# DATA_ROOT points to the externally stored BraTS dataset.
# The medical imaging dataset is deliberately kept outside the
# repository so that large source data are never committed to Git.

PROJECT_ROOT = Path(
    "/mnt/c/Users/USER/OneDrive/Documents/GitHub/glioma-mri-segmentation"
)

DATA_ROOT = Path(
    "/home/corvus_splendens/datasets/brats2024-gli"
)

TRAINING_DATA_ROOT = (
    DATA_ROOT
    / "training"
    / "training_data1_v2"
)

print("Project root exists:", PROJECT_ROOT.exists())
print("Dataset root exists:", DATA_ROOT.exists())
print("Training data exists:", TRAINING_DATA_ROOT.exists())

Project root exists: True
Dataset root exists: True
Training data exists: True


## 2. Dataset Discovery

Each BraTS training case is stored in a separate directory identified by a
BraTS case identifier.

The available case directories are discovered programmatically rather than
manually enumerated. Sorting the directories ensures deterministic ordering
before subsequent dataset partitioning.

In [3]:
# -------------------------------------------------------------
# DISCOVER BRATS TRAINING CASES
# -------------------------------------------------------------
#
# Identify all directories corresponding to BraTS glioma cases.
#
# Sorting ensures that the initial ordering is deterministic
# rather than dependent on filesystem ordering.

case_directories = sorted(
    path
    for path in TRAINING_DATA_ROOT.iterdir()
    if path.is_dir()
    and path.name.startswith("BraTS-GLI-")
)

print(
    "Number of BraTS training cases:",
    len(case_directories)
)

print("\nFirst five cases:")

for case_directory in case_directories[:5]:
    print(case_directory.name)

Number of BraTS training cases: 1350

First five cases:
BraTS-GLI-00005-100
BraTS-GLI-00005-101
BraTS-GLI-00006-100
BraTS-GLI-00006-101
BraTS-GLI-00008-100
